In [2]:
# ================================================================
# POTATO CNN — MODEL TRAINING
# ================================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)
from sklearn.utils.class_weight import compute_class_weight

print("=" * 70)
print("POTATO CNN — MODEL TRAINING")
print("=" * 70)

# ------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

PREPROCESS_FILE = os.path.join(
    PROJECT_DIR,
    "3rd Preprocessing",
    "potato_processed_data.npz"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "6th Trained_Model"
)

os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "potato_cnn_combined_best.keras"
)

print("\nPreprocessed data:")
print(PREPROCESS_FILE)

print("\nModel save location:")
print(MODEL_PATH)


# ------------------------------------------------
# 2. LOAD PREPROCESSED DATA
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOADING POTATO PREPROCESSED DATA")
print("=" * 70)

data = np.load(PREPROCESS_FILE)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)
print("X_test  :", X_test.shape)


# ------------------------------------------------
# 3. CLASS NAMES
# ------------------------------------------------

class_names = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

print("\nClasses:")
for i, name in enumerate(class_names):
    print(f"{i} → {name}")


# ------------------------------------------------
# 4. CLASS WEIGHTS
# ------------------------------------------------

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(c): float(w)
    for c, w in zip(classes, weights)
}

print("\nClass weights:")
for c, w in class_weights.items():
    print(f"{class_names[c]:15s}: {w:.4f}")


# ------------------------------------------------
# 5. BUILD CNN
# ------------------------------------------------

print("\n" + "=" * 70)
print("BUILDING CNN MODEL")
print("=" * 70)

model = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Training augmentation
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),

    # CNN Block 1
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    # CNN Block 2
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    # CNN Block 3
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    # Classification
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),

    # 3 classes
    layers.Dense(3, activation="softmax")
])


# ------------------------------------------------
# 6. COMPILE
# ------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nCNN model created successfully.")


# ------------------------------------------------
# 7. CALLBACKS
# ------------------------------------------------

callbacks = [

    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]


# ------------------------------------------------
# 8. TRAIN
# ------------------------------------------------

print("\n" + "=" * 70)
print("STARTING POTATO CNN TRAINING")
print("=" * 70)

history = model.fit(

    X_train,
    y_train,

    validation_data=(X_val, y_val),

    epochs=25,
    batch_size=32,

    class_weight=class_weights,

    callbacks=callbacks,

    verbose=1
)


# ------------------------------------------------
# 9. LOAD BEST MODEL
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOADING BEST SAVED MODEL")
print("=" * 70)

best_model = tf.keras.models.load_model(MODEL_PATH)

print("✅ Best model loaded successfully")


# ------------------------------------------------
# 10. TRAINING SUMMARY
# ------------------------------------------------

best_train_acc = max(history.history["accuracy"])
best_val_acc = max(history.history["val_accuracy"])

print("\n" + "=" * 70)
print("POTATO CNN TRAINING COMPLETED")
print("=" * 70)

print(f"Best Training Accuracy   : {best_train_acc * 100:.2f}%")
print(f"Best Validation Accuracy : {best_val_acc * 100:.2f}%")

print("\nModel saved at:")
print(MODEL_PATH)

print("\n" + "=" * 70)
print("NEXT STEP → MODEL EVALUATION")
print("=" * 70)

POTATO CNN — MODEL TRAINING
Mounted at /content/drive

Preprocessed data:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/potato_processed_data.npz

Model save location:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/potato_cnn_combined_best.keras

LOADING POTATO PREPROCESSED DATA
X_train : (3561, 224, 224, 3)
X_val   : (445, 224, 224, 3)
X_test  : (446, 224, 224, 3)

Classes:
0 → Healthy
1 → Early_Blight
2 → Late_Blight

Class weights:
Healthy        : 1.2888
Early_Blight   : 1.1413
Late_Blight    : 0.7419

BUILDING CNN MODEL

CNN model created successfully.

STARTING POTATO CNN TRAINING
Epoch 1/25
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5430 - loss: 0.8861
Epoch 1: val_accuracy improved from None to 0.88989, saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/potato_cnn_combined_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Plant D